<div style="background-color: #1e1e1e; padding: 25px; border-radius: 8px; border-left: 8px solid #00ff00; color: #d4d4d4; font-family: 'Courier New', Courier, monospace;">
  <h1 style="color: #00ff00; margin-top: 0;">~$ Lab 13: O Arquiteto de Pipelines</h1>
  <p>> Status: <span style="color: #ff9800;">Vulnerabilidade identificada nas dependências do modelo legado.</span></p>
  <br>
  <b>A MISSÃO:</b>
  <ul style="list-style-type: none; padding-left: 0;">
    <li>[ ] <b>Análise do Caos:</b> Examinar o bloco de código legado (variáveis temporárias e risco de vazamento).</li>
    <li>[ ] <b>Montagem:</b> Instanciar as peças modulares (SimpleImputer e StandardScaler).</li>
    <li>[ ] <b>Encadeamento:</b> Construir a classe Pipeline passando a lista de tuplas.</li>
    <li>[ ] <b>Execução Limpa:</b> Rodar o fluxo do imputador ao scaler de ponta a ponta.</li>
    <li>[ ] <b>Aquecimento Real:</b> Aplicar Arquitetura Pipeline no Dataset do Titanic.</li>
    <li>[ ] <b>Trabalho de Casa:</b> Pipeline Autônomo e Questão Discursiva.</li>
  </ul>
</div>

## 🕵️‍♂️ 1. Análise do Caos: O Código Legado
Observe o bloco experimental que herdamos das extrações passadas. Ele tem imputações esparsas, criação excessiva de DataFrames provisórios, e um perigo iminente caso você esqueça de executar alguma célula intermediária.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# -----------------------------------------------------
# PREPARATIVOS (Mock de Base de Dados)
# -----------------------------------------------------
np.random.seed(42)
n_samples = 500

df_raw = pd.DataFrame({
    'idade': np.random.normal(35, 10, n_samples),
    'salario': np.random.normal(5000, 2000, n_samples),
    'score_credito': np.random.normal(600, 100, n_samples),
    'target_comprou': np.random.choice([0, 1], n_samples)
})

# Sujeira proposital
df_raw.loc[np.random.choice(n_samples, 40, replace=False), 'idade'] = np.nan
df_raw.loc[np.random.choice(n_samples, 20, replace=False), 'salario'] = np.nan

X_raw = df_raw.drop('target_comprou', axis=1)
y = df_raw['target_comprou']

X_train, X_test, y_train, y_test = train_test_split(X_raw, y, test_size=0.2, random_state=42)

# -----------------------------------------------------
# 🔴 O CAOS (CÓDIGO LEGADO)
# -----------------------------------------------------
print("--- FLUXO LEGADO ---")
# Etapa 1: Imputação
imputer_legado = SimpleImputer(strategy='median')
X_train_imputed_array = imputer_legado.fit_transform(X_train)
X_test_imputed_array = imputer_legado.transform(X_test)

# Etapa 2: Transformando de Volta em DF Opcional 
df_train_temp = pd.DataFrame(X_train_imputed_array, columns=X_train.columns)
df_test_temp = pd.DataFrame(X_test_imputed_array, columns=X_test.columns)

# Etapa 3: Standard Scaler
scaler_legado = StandardScaler()
X_train_scaled_array = scaler_legado.fit_transform(df_train_temp)
X_test_scaled_array = scaler_legado.transform(df_test_temp)

# Resultado Final Retornado
df_train_legado = pd.DataFrame(X_train_scaled_array, columns=X_train.columns)
display(df_train_legado.head())

## ⚙️ 2. Montagem das Peças
No novo planejamento arquitetônico, instanciamos nossos operários (Transformadores do Scikit-Learn) separadamente. Nada é executado agora, apenas declaramos!

In [ ]:
# Instanciando nossas peças - a 'caixa de ferramentas' do arquiteto
imputador = SimpleImputer(strategy='median')
escalonador = StandardScaler()

## 🔗 3. Encadeamento com `Pipeline`

Para conectar as peças, utilizaremos a classe mágica `Pipeline`. Inserimos uma *Lista* contendo *Tuplas* em ordem processual.
A estrutura da Tupla é: `('nome_identificador_da_etapa', objeto_instanciado)`.


In [ ]:
# Construindo nossa primeira máquina linear de tratamento:
pipeline_engenharia = Pipeline([
    ('tratamento_nulos', imputador),
    ('padronizacao_escala', escalonador)
])

# Extra: Para visualizar nossa máquina construída como um diagrama
from sklearn import set_config
set_config(display='diagram')

# Retornando o pipeline visualmente
pipeline_engenharia

## 🚀 4. Execução Limpa
O momento da substituição no processo produtivo. Não faremos `.fit` isolados, nem DataFrames intermediários para `imputer` ou `scaler`.

In [ ]:
# Enviando a base imaculada do Train e Test para o Tubo (Pipeline)
X_train_clean_array = pipeline_engenharia.fit_transform(X_train)

# No Teste, zero variáveis espalhadas, usamos um singelo método:
X_test_clean_array  = pipeline_engenharia.transform(X_test)

print("✅ FLUXO DE PRODUÇÃO [SUCESSO] ✅\n")

# Visualizando o resultado final das mesmas estatísticas transformadas
X_train_clean_df = pd.DataFrame(X_train_clean_array, columns=X_train.columns)
display(X_train_clean_df.head())


> **Status Final: Missões Completadas** 
> - Intervenções manuais: zeradas.
> - Riscos de inconsistência de Leakage (`.fit` soltos no Test): neutralizados.

---
## 🏋️ 5. Aquecimento: O Seu Primeiro Pipeline Real
Chegou a sua vez! Vamos utilizar o famoso dataset do **Titanic**, que contém diversos valores ausentes na coluna `age` e dados em diferentes escalas (`fare` e `age`).

**Sua missão:**
1. Carregar os dados (já fornecido abaixo).
2. O código já vai separar as variáveis numéricas `age` e `fare`.
3. Criar o seu próprio Pipeline (`pipeline_titanic`) contendo:
   - Um Imputador (`SimpleImputer` com a estratégia que preferir).
   - Um Escalonador (`StandardScaler` ou `MinMaxScaler`).
4. Treinar (`fit_transform`) nos dados e visualizar o resultado.

In [ ]:
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Carregando o dataset real da internet
print("Baixando dados reais do Titanic...")
titanic = sns.load_dataset('titanic')

# Selecionamos apenas as colunas numéricas que precisamos
df_titanic_num = titanic[['age', 'fare']].copy()

print("\nValores ausentes originais no formato original:")
display(df_titanic_num.isnull().sum())

# ==========================================
# 👨‍💻 ESCREVA SEU CÓDIGO AQUI
# ==========================================
# 1. Instancie as ferramentas
meu_imputador = SimpleImputer(strategy='median')
meu_escalonador = StandardScaler()

# 2. Construa a esteira do Pipeline
pipeline_titanic = Pipeline([
    ('imputacao_nulos', meu_imputador),
    ('padronizacao_escala', meu_escalonador)
])

# 3. Envie os dados para a esteira funcionar!
df_titanic_limpo_array = pipeline_titanic.fit_transform(df_titanic_num)

# 4. Veja o resultado limpo
pd.DataFrame(df_titanic_limpo_array, columns=df_titanic_num.columns).head()

### 🚀 5.1 BÔNUS: O Gran Finale - Consumindo os Dados Gerados
*Nota: Execute esta célula apenas após rodar o item 5!*

Agora que a sua variável limpa no Pipeline está maravilhosa, para onde ela vai? Para um algoritmo de Machine Learning alimentá-la, é claro!
Abaixo, instanciamos um modelo clássico de **Regressão Logística** para prever quem sobreviveu ao Titanic, consumindo o array transformado que acabou de sair da nossa 'esteira'. 

Isso prova a vantagem gigantesca de não deixarmos variáveis soltas!

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# 1. Nosso alvo estatístico clássico (survived: 0 = Não, 1 = Sim)
y_titanic = titanic['survived']

# 2. Instanciamos o algoritmo
modelo_classificacao = LogisticRegression(random_state=42)

# 3. Treinamos o preditor DIRETAMENTE com o array limpo gerado pelo SEU Pipeline criado no notebook acima!
modelo_classificacao.fit(df_titanic_limpo_array, y_titanic)

# 4. Medindo o impacto da nossa limpeza:
predicoes = modelo_classificacao.predict(df_titanic_limpo_array)
acuracia = accuracy_score(y_titanic, predicoes)
print(f"🎯 Acurácia Base do nosso modelo preditivo: {acuracia:.2%}\n")

# =========================================================
# 🤯 O VERDADEIRO PODER DO PIPELINE NO MERCADO (SPOILER):
# O Scikit-Learn permite colocar o Próprio Modelo Preditivo DENTRO da esteira!
# =========================================================
pipeline_completo = Pipeline([
    ('meu_pre_processamento_inteiro', pipeline_titanic), # Nosso pipeline de limpeza do item 5 entra listado como se fosse APENAS uma etapa!
    ('meu_modelo_preditivo', LogisticRegression(random_state=42)) # O modelo é literalmente a última máquina mecânica do tubo.
])

# O Pipeline engole o dado bruto de uma vez, limpa, ajusta escalas e Treina o modelo... Tudo num único '.fit()'! Mágico e Blindado contra Leakage.
pipeline_completo.fit(df_titanic_num, y_titanic)

print("Previsões das 5 primeiras pessoas feitas pelo nosso Super Pipeline Master (Dados brutos -> Previsão):")
display(pipeline_completo.predict(df_titanic_num.head(5)))

---
## 🏠 6. Trabalho Para Casa: O Arquiteto Independente
Chegou o momento de você consolidar o conhecimento operando sem a minha ajuda passo-a-passo.

Para este desafio, utilizaremos o excelente dataset **Penguins** (Pinguins), que cataloga características físicas de diferentes espécies de pinguins isolados em ilhas da Antártida. 
No processamento ambiental real, muitos pinguins escaparam da balança e os pesquisadores não conseguiram medi-los corretamente, gerando dados nulos na tabela base.

**O Desafio Prático:**
1. Abaixo eu iniciei a análise com duas características vitais: `body_mass_g` (peso em gramas) e `flipper_length_mm` (tamanho da nadadeira).
2. Instancie e crie do zero um novo **Pipeline** (`pipeline_pinguins`) que impute os nulos e padronize a escala.
3. Exiba o resultado final em um *Dataframe* limpo chamado `pinguins_limpo_df`.


In [ ]:
# Carregando o dataset real da internet
pinguins = sns.load_dataset('penguins')

# Isolando numéricas com nulos ambientais
df_hw_num = pinguins[['body_mass_g', 'flipper_length_mm']].copy()

print("Valores ausentes da missão:")
display(df_hw_num.isnull().sum())

# ==========================================
# 👨‍💻 INICIE SEU TRABALHO APÓS ESSA LINHA
# ==========================================

pipeline_pinguins = Pipeline([
    ('imputacao', SimpleImputer(strategy='median')),
    ('escalonamento', StandardScaler())
])

pinguins_limpo_array = pipeline_pinguins.fit_transform(df_hw_num)
pinguins_limpo_df = pd.DataFrame(pinguins_limpo_array, columns=df_hw_num.columns)

print("\nNulos após pipeline:")
print(pinguins_limpo_df.isnull().sum().sum())
display(pinguins_limpo_df.head())

### ✍️ Missão Final: Questão Discursiva
Reflita sobre o fluxo frágil que presenciamos no início desta aula (processos manuais repetidos e soltos) comparado à sofisticação da arquitetura de `Pipeline`.

**Responda com suas palavras:** *Se você já treinou um modelo para prever a espécie de pinguins e ele fosse ao ar [em produção], na manhã seguinte, seu chefe te enviaria um arquivo do Excel com os dados de apenas **UM** pinguim novo para ser analisado.  
Explique por que pegar esse único pinguim e processá-lo com um método como `pipeline.transform(dado_novo)` é infinitamente mais seguro e confiável do que tentar aplicar `Imputers` e `Scalers` separadamente em objetos soltos no script? Ademais, o que é o famigerado Data Leakage e como o Pipeline nos blinda disso?*

*** 
*(Dê um **clique duplo** sobre este texto abaixo e sobrescreva-o com a sua resposta. Em seguida, pressione `Shift + Enter` para finalizar e deixe salvo no notebook!)*

> **Minha Resposta:** 
> 
> [ Escreva seu texto aqui... ]
